# Muse Spark as a gate on audio

Nov 1. Can Muse Spark listen to a separated sound and say whether it is right? 388 calls, $0.176.

**Data:** AlbumDB stems, sent to the Meta API in contributor mode.

**What it decided:** no. It rejected 13 of 13 wrong outputs and 0 of 8 right ones, but 18 of 125 clips stayed undecided and it refused many requests. Muse Spark names images and writes text in the product; it does not judge audio.

This is a selection of the cells from that work, with the output they printed. They rely on earlier cells and on a Drive folder of datasets and outputs that is not in this repository.


## At scale, with refusals detected and retried


In [ ]:
"""V3: Muse Spark as a target-specific gate, at scale, with refusals detected and retried.

Frozen before running. Gate rule: reject at least 80% of wrong SAM outputs, wrongly reject at most
10% of correct ones, and leave at most 10% of clips undecided after three attempts.
Production's refusal markers are recorded beside the broadened ones, to count what they miss.
"""

from google.colab import userdata

VERIFY = r'''
import hashlib
import json
import re
import sys
import tempfile
import time
import urllib.error
import urllib.request
import base64
import os
import subprocess
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

RATE = 48000
URL = "https://api.meta.ai/v1/responses"
MODEL = "muse-spark-1.2-contributor"
MAX_OUTPUT_TOKENS = 4000
ATTEMPTS_PER_CLIP = 3
CALL_LIMIT = 200
ABSENT_DB = -25.0
STEMS = ("vocals", "drums", "bass", "guitar", "piano", "other")
PROMPT = ("This is a music clip. Name the single most prominent instrument or sound you hear, "
          "as a short noun phrase such as 'electric bass' or 'lead vocals'. "
          "Reply with that phrase only.")
# engine/suggest.py DEAF_MARKERS as shipped today.
PRODUCTION_MARKERS = ("waveform image only", "waveform only", "only a waveform", "just a waveform",
                      "silent waveform", "waveform with no audio", "waveform without audio",
                      "no audible", "no audio", "cannot hear", "can't hear", "unable to hear",
                      "black background", "black screen", "text-based", "silent display")
# Phrasings observed in this morning's 56 calls that the list above let through.
OBSERVED_MARKERS = ("ability to hear", "ability to listen", "ability to play", "ability to process",
                    "waveform image", "waveform visualization", "waveform images",
                    "image of a waveform", "only see", "can't identify", "cannot identify",
                    "not able to hear", "unable to listen", "can't listen", "cannot listen")
LONGEST_PHRASE_WORDS = 8
# Mirrors engine/fallback.py PROMPT_STEM_KEYWORDS after today's routing change.
KEYWORDS = (
    ("vocals", {"vocal", "vocals", "voice", "sing", "singing", "singer"}),
    ("drums", {"drum", "drums", "kick", "snare", "hihat", "hat", "cymbal"}),
    ("bass", {"bass", "sub", "subbass"}),
    ("guitar", {"guitar", "guitars", "ukulele", "ukelele", "banjo"}),
    ("piano", {"piano", "pianos"}),
    ("other", {"synth", "synthesiser", "synthesizer", "pad", "lead", "supersaw", "pluck",
               "organ", "rhodes", "wurlitzer", "harpsichord", "accordion", "keys", "keyboard",
               "string", "strings", "cello", "violin", "viola", "harp", "brass", "horn",
               "horns", "trumpet", "trombone", "sax", "saxophone", "flute", "clarinet", "oboe",
               "woodwind"}),
)
TOKEN = re.compile(r"[a-z]+")
state = {"calls": 0, "input_tokens": 0, "output_tokens": 0}


def category(text):
    tokens = set(TOKEN.findall(text.lower()))
    for stem, words in KEYWORDS:
        if tokens & words:
            return stem
    return None


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    x = audio.mean(axis=1)
    if rate != RATE:
        x = librosa.resample(x, orig_sr=rate, target_sr=RATE)
    x = np.asarray(x, dtype=np.float64)
    return x if length is None else x[:length]


def si_sdr(ref, est):
    n = min(len(ref), len(est))
    t, a = ref[:n] - ref[:n].mean(), est[:n] - est[:n].mean()
    p = float(t @ t)
    if p < 1e-12:
        return float("-inf")
    proj = t * float(a @ t) / p
    err = a - proj
    return 10 * np.log10((proj @ proj + 1e-12) / (err @ err + 1e-12))


def post(payload, key):
    """Same transport rule as engine/meta_api.post_responses: retry transport errors and 5xx."""
    body = json.dumps(payload).encode()
    for _ in range(3):
        request = urllib.request.Request(URL, data=body, method="POST", headers={
            "Authorization": f"Bearer {key}", "Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=300) as response:
                return json.load(response)
        except urllib.error.HTTPError as error:
            if error.code < 500:
                raise RuntimeError(f"Meta API returned HTTP {error.code}") from None
        except urllib.error.URLError:
            pass
        time.sleep(2)
    raise RuntimeError("Meta API unreachable after 3 attempts")


def output_text(reply):
    chunks = []
    for item in reply.get("output", []) or []:
        for part in (item.get("content") or []) if isinstance(item, dict) else []:
            if isinstance(part, dict) and isinstance(part.get("text"), str):
                chunks.append(part["text"])
    return "\n".join(chunks).strip()


def ask(samples, key, scratch):
    """Up to three attempts; returns every reply with how each detector judged it."""
    peak = float(np.max(np.abs(samples)))
    wav, mp4 = scratch / "clip.wav", scratch / "clip.mp4"
    sf.write(str(wav), (samples * (0.891 / peak) if peak > 0 else samples).astype(np.float32),
             RATE, subtype="PCM_16")
    subprocess.run(["ffmpeg", "-y", "-i", str(wav), "-filter_complex",
                    "[0:a]showwaves=s=320x180:mode=line:rate=5[v]", "-map", "[v]", "-map", "0:a",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", str(mp4)],
                   check=True, capture_output=True)
    video = "data:video/mp4;base64," + base64.b64encode(mp4.read_bytes()).decode()
    payload = {"model": MODEL, "max_output_tokens": MAX_OUTPUT_TOKENS, "input": [{
        "role": "user", "content": [{"type": "input_text", "text": PROMPT},
                                    {"type": "input_video", "video_url": video}]}]}
    attempts = []
    for _ in range(ATTEMPTS_PER_CLIP):
        if state["calls"] >= CALL_LIMIT:
            break
        state["calls"] += 1
        reply = post(payload, key)
        usage = reply.get("usage") or {}
        state["input_tokens"] += int(usage.get("input_tokens") or 0)
        state["output_tokens"] += int(usage.get("output_tokens") or 0)
        text = output_text(reply)
        lowered = text.lower()
        production_refused = any(m in lowered for m in PRODUCTION_MARKERS)
        broadened_refused = (production_refused
                             or any(m in lowered for m in OBSERVED_MARKERS)
                             or len(text.split()) > LONGEST_PHRASE_WORDS)
        attempts.append({"text": text, "production_refused": production_refused,
                         "broadened_refused": broadened_refused})
        if not broadened_refused:
            break
    decided = next((a["text"] for a in attempts if not a["broadened_refused"]), None)
    return {"attempts": attempts, "answer": decided,
            "category": category(decided) if decided else None}


def digest(path):
    return hashlib.sha256(path.read_bytes()[: 1 << 20]).hexdigest()


def main():
    key = os.environ["META_API_KEY"]
    sam_runs, bank_runs = sys.argv[1].split(","), sys.argv[2].split(",")
    out = Path(sys.argv[3])
    sam_rows, bank_rows = [], []
    with tempfile.TemporaryDirectory() as tmp:
        scratch = Path(tmp)
        for run_path in sam_runs:
            run = Path(run_path)
            config = json.loads((run / "config.json").read_text())
            for case in config["cases"]:
                case_dir = run / case["id"]
                candidate = case_dir / "sam-audio-large" / "candidate-0.wav"
                if not candidate.is_file():
                    continue
                n = len(mono(case_dir / "reference.wav"))
                stems = {s: mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{s}.wav", n)
                         for s in STEMS}
                output = mono(candidate, n)
                truth = max(STEMS, key=lambda s: si_sdr(stems[s], output))
                result = ask(output, key, scratch)
                sam_rows.append({"run": run.name, "case": case["id"], "prompt": case["prompt"],
                                 "requested": category(case["prompt"]), "actually": truth,
                                 **result})
                print(f"  SAM  {case['id']:22s} asked {case['prompt']!r:24s} is {truth:7s} "
                      f"heard {result['answer']!r} in {len(result['attempts'])}", flush=True)
        seen = set()
        for run_path in bank_runs:
            run = Path(run_path)
            for case in json.loads((run / "config.json").read_text())["cases"]:
                case_dir = run / case["id"]
                if not (case_dir / "mix.wav").is_file():
                    continue
                signature = digest(case_dir / "mix.wav")
                if signature in seen:
                    continue
                seen.add(signature)
                mixture = mono(case_dir / "mix.wav")
                mix_rms = float(np.sqrt(np.mean(mixture ** 2)))
                for stem in STEMS:
                    path = case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{stem}.wav"
                    if not path.is_file():
                        continue
                    samples = mono(path, len(mixture))
                    level = 20 * np.log10(max(float(np.sqrt(np.mean(samples ** 2))), 1e-12)
                                          / max(mix_rms, 1e-12))
                    if level < ABSENT_DB:
                        continue
                    result = ask(samples, key, scratch)
                    bank_rows.append({"run": run.name, "mix": case["id"], "stem": stem,
                                      "level_db": float(level), **result})
                    print(f"  BANK {case['id']:12s} {stem:7s} {level:+6.1f} dB heard "
                          f"{result['answer']!r} in {len(result['attempts'])}", flush=True)
                    if state["calls"] >= CALL_LIMIT:
                        break
    out.write_text(json.dumps({"sam": sam_rows, "bank": bank_rows, "usage": state}, indent=2))
    print("VERIFY_USAGE " + json.dumps(state))


if __name__ == "__main__":
    main()
'''
Path("/content/verify.py").write_text(VERIFY)
verify_env = dict(ENV)
verify_env["CUDA_VISIBLE_DEVICES"] = "-1"
verify_env["META_API_KEY"] = userdata.get("META_API_KEY")
if not verify_env["META_API_KEY"]:
    raise RuntimeError("META_API_KEY Colab secret is empty")
sam_runs = ",".join(str(BASE / "runs" / n) for n in ("smoke-001", "dev-002", "dev-003", "holdout-004"))
bank_runs = ",".join(str(BASE / "runs" / n) for n in ("holdout-004", "voxbass-006"))
command = (f"{UV} run --no-sync --project {PROJECTS['pitch']} python /content/verify.py "
           f"{sam_runs} {bank_runs} /content/verify.json; echo VERIFY_DONE")
with open("/content/verify.log", "w") as handle:
    job = subprocess.Popen(["bash", "-c", command], env=verify_env, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
del verify_env
Path("/content/verify.pid").write_text(str(job.pid))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = "/content/verify.log", "/content/verify.pid", "VERIFY_DONE"
print(f"verification running as pid {job.pid}; key passed to the subprocess only, never printed")


verification running as pid 61588; key passed to the subprocess only, never printed


In [ ]:
"""V3 results against the frozen gate rule. Clips never asked (the call cap) are excluded."""

result = json.loads(Path("/content/verify.json").read_text())
sam_rows = [r for r in result["sam"] if r["attempts"]]
bank_rows = [r for r in result["bank"] if r["attempts"]]
attempts = [a for r in sam_rows + bank_rows for a in r["attempts"]]

refused = [a for a in attempts if a["broadened_refused"]]
missed = [a for a in refused if not a["production_refused"]]
print("refusal detection, per call")
print(f"  calls {len(attempts)}, refusals {len(refused)} ({100 * len(refused) / len(attempts):.0f}%)")
print(f"  refusals production's DEAF_MARKERS would have accepted as an answer: "
      f"{len(missed)}/{len(refused)}")

clips = sam_rows + bank_rows
undecided = [r for r in clips if r["answer"] is None]
first_try = sum(1 for r in clips if r["answer"] is not None and len(r["attempts"]) == 1)
print("\nper clip, up to three attempts")
print(f"  clips {len(clips)}, decided {len(clips) - len(undecided)}, "
      f"undecided {len(undecided)} ({100 * len(undecided) / len(clips):.0f}%)")
print(f"  decided on the first attempt {first_try}")

print("\nidentification accuracy on Demucs stems, when it answered")
by_stem = {}
for row in bank_rows:
    if row["answer"] is None:
        by_stem.setdefault(row["stem"], [0, 0, 0])[2] += 1
        continue
    hit = row["category"] == row["stem"]
    entry = by_stem.setdefault(row["stem"], [0, 0, 0])
    entry[0] += int(hit)
    entry[1] += 1
total_hit = sum(v[0] for v in by_stem.values())
total_answered = sum(v[1] for v in by_stem.values())
for stem, (hit, answered, silent) in sorted(by_stem.items()):
    print(f"  {stem:7s} correct {hit:2d}/{answered:2d}   undecided {silent}")
print(f"  {'all':7s} correct {total_hit}/{total_answered} "
      f"({100 * total_hit / max(total_answered, 1):.1f}%)")
wrong_ids = [r for r in bank_rows if r["answer"] and r["category"] != r["stem"]]
for row in wrong_ids:
    print(f"    misidentified {row['mix']} {row['stem']} ({row['level_db']:+.1f} dB) as "
          f"{row['answer']!r}")

print("\ngate on the SAM path: accept only when what it hears matches what was asked")
wrong = [r for r in sam_rows if r["actually"] != r["requested"]]
right = [r for r in sam_rows if r["actually"] == r["requested"]]
wrong_rejected = sum(1 for r in wrong if r["answer"] is not None and r["category"] != r["requested"])
wrong_undecided = sum(1 for r in wrong if r["answer"] is None)
right_rejected = sum(1 for r in right if r["answer"] is not None and r["category"] != r["requested"])
right_undecided = sum(1 for r in right if r["answer"] is None)
print(f"  wrong outputs {len(wrong)}: rejected {wrong_rejected}, undecided {wrong_undecided}, "
      f"let through {len(wrong) - wrong_rejected - wrong_undecided}")
print(f"  right outputs {len(right)}: wrongly rejected {right_rejected}, "
      f"undecided {right_undecided}")
reject_rate = wrong_rejected / max(len(wrong), 1)
false_rate = right_rejected / max(len(right), 1)
undecided_rate = len(undecided) / len(clips)
print(f"\n  rule: wrong rejected >= 80%      -> {100 * reject_rate:.0f}%  "
      f"{'met' if reject_rate >= 0.8 else 'FAILED'}")
print(f"  rule: right rejected <= 10%      -> {100 * false_rate:.0f}%  "
      f"{'met' if false_rate <= 0.1 else 'FAILED'}")
print(f"  rule: undecided clips <= 10%     -> {100 * undecided_rate:.0f}%  "
      f"{'met' if undecided_rate <= 0.1 else 'FAILED'}")
usage = result["usage"]
cost = usage["input_tokens"] * 0.10 / 1e6 + usage["output_tokens"] * 0.20 / 1e6
print(f"\n  {usage['calls']} calls, ${cost:.3f} at contributor rates, "
      f"${cost / usage['calls']:.5f} per call")


refusal detection, per call
  calls 200, refusals 93 (46%)
  refusals production's DEAF_MARKERS would have accepted as an answer: 66/93

per clip, up to three attempts
  clips 125, decided 107, undecided 18 (14%)
  decided on the first attempt 80

identification accuracy on Demucs stems, when it answered
  bass    correct 20/20   undecided 6
  drums   correct 20/20   undecided 4
  guitar  correct 20/20   undecided 1
  other   correct  0/ 0   undecided 5
  piano   correct  0/ 1   undecided 1
  vocals  correct 25/25   undecided 1
  all     correct 85/86 (98.8%)
    misidentified karma_piano piano (-17.1 dB) as 'electric guitar'

gate on the SAM path: accept only when what it hears matches what was asked
  wrong outputs 13: rejected 13, undecided 0, let through 0
  right outputs 8: wrongly rejected 0, undecided 0

  rule: wrong rejected >= 80%      -> 100%  met
  rule: right rejected <= 10%      -> 0%  met
  rule: undecided clips <= 10%     -> 14%  FAILED

  200 calls, $0.087 at contribut

## The undecided clips, and what a picture changes


In [ ]:
"""V4: re-ask only the 18 undecided clips, with production's own LISTEN_INSTRUCTION and 5 attempts.

Separates two explanations: refusals that are random per call (five tries should clear almost
all of them) from refusals that are systematic for some content (they stay undecided).
Prose replies are judged by refusal markers only, and categorised by the first instrument named.
"""

RETRY = r'''
import importlib.util
import json
import re
import sys
import tempfile
import os
import subprocess
import base64
from pathlib import Path

import numpy as np
import soundfile as sf

spec = importlib.util.spec_from_file_location("verify", "/content/verify.py")
verify = importlib.util.module_from_spec(spec)
spec.loader.exec_module(verify)

LISTEN_INSTRUCTION = (
    "This is a music clip. Listen to it and describe, in plain prose, each distinct sound you can "
    "hear - instruments, voices, drums, textures. For each one say what it is and how it behaves "
    "in the mix. Then give one sentence describing the clip as a whole: its genre, tempo feel and "
    "mood. Be concrete and only name sounds you actually hear."
)
ATTEMPTS = 5
CALL_LIMIT = 90
WORD = re.compile(r"[a-z]+")
state = {"calls": 0, "input_tokens": 0, "output_tokens": 0}


def first_category(text):
    """The category of the earliest instrument word, since prose lists the dominant sound first."""
    for word in WORD.findall(text.lower()):
        for stem, words in verify.KEYWORDS:
            if word in words:
                return stem
    return None


def refused(text):
    lowered = text.lower()
    return (any(m in lowered for m in verify.PRODUCTION_MARKERS)
            or any(m in lowered for m in verify.OBSERVED_MARKERS))


def main():
    key = os.environ["META_API_KEY"]
    previous = json.loads(Path("/content/verify.json").read_text())
    pending = [r for r in previous["bank"] if r["attempts"] and r["answer"] is None]
    runs = Path(sys.argv[1])
    rows = []
    with tempfile.TemporaryDirectory() as tmp:
        scratch = Path(tmp)
        for row in pending:
            case_dir = runs / row["run"] / row["mix"]
            n = len(verify.mono(case_dir / "mix.wav"))
            samples = verify.mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{row['stem']}.wav", n)
            peak = float(np.max(np.abs(samples)))
            wav, mp4 = scratch / "clip.wav", scratch / "clip.mp4"
            sf.write(str(wav), (samples * (0.891 / peak)).astype(np.float32), verify.RATE,
                     subtype="PCM_16")
            subprocess.run(["ffmpeg", "-y", "-i", str(wav), "-filter_complex",
                            "[0:a]showwaves=s=320x180:mode=line:rate=5[v]", "-map", "[v]",
                            "-map", "0:a", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a",
                            "aac", str(mp4)], check=True, capture_output=True)
            video = "data:video/mp4;base64," + base64.b64encode(mp4.read_bytes()).decode()
            payload = {"model": verify.MODEL, "max_output_tokens": verify.MAX_OUTPUT_TOKENS,
                       "input": [{"role": "user", "content": [
                           {"type": "input_text", "text": LISTEN_INSTRUCTION},
                           {"type": "input_video", "video_url": video}]}]}
            tries, answer = 0, None
            while tries < ATTEMPTS and state["calls"] < CALL_LIMIT:
                tries += 1
                state["calls"] += 1
                reply = verify.post(payload, key)
                usage = reply.get("usage") or {}
                state["input_tokens"] += int(usage.get("input_tokens") or 0)
                state["output_tokens"] += int(usage.get("output_tokens") or 0)
                text = verify.output_text(reply)
                if not refused(text):
                    answer = text
                    break
            heard = first_category(answer) if answer else None
            rows.append({"mix": row["mix"], "stem": row["stem"], "level_db": row["level_db"],
                         "tries": tries, "answer": answer, "heard": heard})
            summary = (answer or "")[:90].replace("\n", " ")
            print(f"  {row['mix']:16s} {row['stem']:7s} {row['level_db']:+6.1f} dB  tries {tries}  "
                  f"heard {heard!s:7s} {summary!r}", flush=True)
    Path("/content/verify_retry.json").write_text(json.dumps({"rows": rows, "usage": state},
                                                             indent=2))
    print("RETRY_USAGE " + json.dumps(state))


if __name__ == "__main__":
    main()
'''
Path("/content/verify_retry.py").write_text(RETRY)
retry_env = dict(ENV)
retry_env["CUDA_VISIBLE_DEVICES"] = "-1"
retry_env["META_API_KEY"] = userdata.get("META_API_KEY")
command = (f"{UV} run --no-sync --project {PROJECTS['pitch']} python /content/verify_retry.py "
           f"{BASE / 'runs'}; echo RETRY_DONE")
with open("/content/verify_retry.log", "w") as handle:
    job = subprocess.Popen(["bash", "-c", command], env=retry_env, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
del retry_env
Path("/content/verify_retry.pid").write_text(str(job.pid))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = ("/content/verify_retry.log", "/content/verify_retry.pid",
                                     "RETRY_DONE")
print(f"re-asking 18 undecided clips as pid {job.pid}")


re-asking 18 undecided clips as pid 75305


In [ ]:
"""Append the Muse Spark findings to today's results file."""

MUSE = """
## Muse Spark as a target-specific verifier (290 calls, $0.157 at contributor rates)

Asked to name the instrument in an isolated stem, sent as an mp4 with a waveform picture, which is
how production already sends audio. Contributor pricing from dev.meta.ai: $0.10 input and $0.20
output per million tokens, about $0.00043 per call.

| measure | result |
|---|---|
| identification accuracy when it answers, Demucs stems | 85 / 86 (98.8%): vocals 25/25, bass 20/20, drums 20/20, guitar 20/20, piano 0/1 |
| gate on the SAM path: wrong outputs rejected | 13 / 13 |
| gate on the SAM path: right outputs wrongly rejected | 0 / 8 |
| clips undecided after three attempts | 18 / 125 (14%), against a 10% bar: **failed** |
| `other` stems (organ, cello, synth) ever answered | 0 / 5 |

The undecided clips are systematic, not unlucky. All 18 were asked five more times with
production's own LISTEN_INSTRUCTION and refused every time: eight refusals in a row each, where a
random 46% refusal rate would give that about 0.2% of the time. They include loud stems, such as
two bass parts at -3.4 and -3.9 dB, so level does not explain them.

Where it answers it is right, which is what Judge could not do: it scored a bass extraction above
the true piano. The SAM-path cases here used prompts that now route to Demucs, so this proves it
catches "SAM returned bass instead", not performance on the prompts SAM still serves.

### A production defect found on the way

93 of 200 calls were refusals, and production's `DEAF_MARKERS` in `engine/suggest.py` would have
accepted 66 of those 93 as a description. Its list has "cannot hear" and "no audio" but not the
phrasings the model actually used: "don't have the ability to hear", "ability to listen",
"waveform image alone", "can't identify". So live suggestion chips and summaries can be built from
a refusal, and the retry meant to catch it never fires.
"""
results_path = BASE / "runs" / "RESULTS-2026-11-01.md"
results_path.write_text(results_path.read_text() + MUSE)
print(f"appended to {results_path}")


appended to /content/drive/MyDrive/mixray-quality/runs/RESULTS-2026-11-01.md


In [ ]:
"""V5: same stem audio and same short prompt as V3, but the picture is the mixture's waveform.

Only the picture changes from V3, where these 18 clips refused three times each. Accuracy is
checked as well, in case a busier picture makes the model describe the band instead of the stem.
"""

PICTURE = r'''
import importlib.util
import json
import os
import subprocess
import sys
import tempfile
import base64
from pathlib import Path

import numpy as np
import soundfile as sf

spec = importlib.util.spec_from_file_location("verify", "/content/verify.py")
verify = importlib.util.module_from_spec(spec)
spec.loader.exec_module(verify)

ATTEMPTS = 3
CALL_LIMIT = 54
state = {"calls": 0, "input_tokens": 0, "output_tokens": 0}


def write(path, samples):
    peak = float(np.max(np.abs(samples)))
    sf.write(str(path), (samples * (0.891 / peak) if peak > 0 else samples).astype(np.float32),
             verify.RATE, subtype="PCM_16")


def main():
    key = os.environ["META_API_KEY"]
    pending = [r for r in json.loads(Path("/content/verify.json").read_text())["bank"]
               if r["attempts"] and r["answer"] is None]
    runs = Path(sys.argv[1])
    rows = []
    with tempfile.TemporaryDirectory() as tmp:
        scratch = Path(tmp)
        for row in pending:
            case_dir = runs / row["run"] / row["mix"]
            mixture = verify.mono(case_dir / "mix.wav")
            n = len(mixture)
            stem = verify.mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{row['stem']}.wav", n)
            picture_wav, sound_wav, mp4 = (scratch / "picture.wav", scratch / "sound.wav",
                                           scratch / "clip.mp4")
            write(picture_wav, mixture)
            write(sound_wav, stem)
            seconds = f"{n / verify.RATE:.3f}"
            subprocess.run(["ffmpeg", "-y", "-i", str(picture_wav), "-i", str(sound_wav),
                            "-filter_complex", "[0:a]showwaves=s=320x180:mode=line:rate=5[v]",
                            "-map", "[v]", "-map", "1:a", "-t", seconds, "-c:v", "libx264",
                            "-pix_fmt", "yuv420p", "-c:a", "aac", str(mp4)],
                           check=True, capture_output=True)
            video = "data:video/mp4;base64," + base64.b64encode(mp4.read_bytes()).decode()
            payload = {"model": verify.MODEL, "max_output_tokens": verify.MAX_OUTPUT_TOKENS,
                       "input": [{"role": "user", "content": [
                           {"type": "input_text", "text": verify.PROMPT},
                           {"type": "input_video", "video_url": video}]}]}
            tries, answer = 0, None
            while tries < ATTEMPTS and state["calls"] < CALL_LIMIT:
                tries += 1
                state["calls"] += 1
                reply = verify.post(payload, key)
                usage = reply.get("usage") or {}
                state["input_tokens"] += int(usage.get("input_tokens") or 0)
                state["output_tokens"] += int(usage.get("output_tokens") or 0)
                text = verify.output_text(reply)
                lowered = text.lower()
                if not (any(m in lowered for m in verify.PRODUCTION_MARKERS)
                        or any(m in lowered for m in verify.OBSERVED_MARKERS)
                        or len(text.split()) > verify.LONGEST_PHRASE_WORDS):
                    answer = text
                    break
            heard = verify.category(answer) if answer else None
            rows.append({"mix": row["mix"], "stem": row["stem"], "level_db": row["level_db"],
                         "tries": tries, "answer": answer, "heard": heard})
            print(f"  {row['mix']:16s} {row['stem']:7s} {row['level_db']:+6.1f} dB  tries {tries}  "
                  f"heard {answer!r}", flush=True)
    Path("/content/verify_picture.json").write_text(json.dumps({"rows": rows, "usage": state},
                                                               indent=2))
    print("PICTURE_USAGE " + json.dumps(state))


if __name__ == "__main__":
    main()
'''
Path("/content/verify_picture.py").write_text(PICTURE)
picture_env = dict(ENV)
picture_env["CUDA_VISIBLE_DEVICES"] = "-1"
picture_env["META_API_KEY"] = userdata.get("META_API_KEY")
command = (f"{UV} run --no-sync --project {PROJECTS['pitch']} python /content/verify_picture.py "
           f"{BASE / 'runs'}; echo PICTURE_DONE")
with open("/content/verify_picture.log", "w") as handle:
    job = subprocess.Popen(["bash", "-c", command], env=picture_env, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
del picture_env
Path("/content/verify_picture.pid").write_text(str(job.pid))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = ("/content/verify_picture.log", "/content/verify_picture.pid",
                                     "PICTURE_DONE")
print(f"18 clips with the mixture's waveform as the picture, pid {job.pid}")


18 clips with the mixture's waveform as the picture, pid 79973


In [ ]:
MUSE_PICTURE = """
### The picture decides whether it listens (42 calls, $0.019)

The 18 clips that had refused eight times in a row were sent again with the same stem audio and the
same short prompt, changing only the picture to the waveform of the original mixture.

| measure | result |
|---|---|
| decided within three attempts | 11 / 18, all 11 correct |
| still undecided | 7 / 18, four of them `other` stems |

It did not start describing the band: every answer named the stem's instrument, and the Karma piano
it had called an electric guitar came back as "piano".

A two-stage policy, stem waveform first and the mixture's waveform for whatever refuses, meets all
three gate conditions on these clips: 13/13 wrong SAM outputs rejected, 0/8 right ones rejected,
7/125 (5.6%) undecided, and 96/97 identifications correct. The second stage was designed after the
first stage's failures were seen, so this is promising rather than established: it needs a run on
clips it has never seen. `other` stays weak at 1/5 answered.

Muse Spark spend today: 332 calls, $0.176.
"""
results_path.write_text(results_path.read_text() + MUSE_PICTURE)
print("appended")


appended
